# Qwable 27B Server

**27B dense** — reasoning + instruction-tuned fine-tune of Qwen3.6-27B by Mia-AiLab.

- Q4_K_M quant: 15.66 GB — **borderline for T4** (15GB VRAM)
- Use 2K context max, or it will OOM
- MTP variant for speculative decoding
- Based on Qwen3.6-27B
- MIT license

**Warning**: This is the tightest fit in the repo. If it OOMs, use a smaller model or try the abliterated version with a smaller quant.

In [ ]:
# Install dependencies
# llama-cpp-python with CUDA 12.4 — no torch needed (saves ~2GB RAM)
!pip install -q llama-cpp-python huggingface_hub --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
import llama_cpp; print(f"llama-cpp-python {llama_cpp.__version__}")
print(f"CUDA GPU offload: {llama_cpp.llama_supports_gpu_offload()}")

In [ ]:
# Download model
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="Mia-AiLab/Qwable-3.6-27b-MTP",
    filename="Qwable-3.6-27b_q4_k_m.gguf"
)
import os
print(f"Model: {os.path.getsize(model_path)/1e9:.2f} GB")
print(f"Path: {model_path}")

# Global vars for server cell
MODEL_NAME = "qwable-27b"
CONTEXT = 2048


In [ ]:
# Check available VRAM (no model loaded yet)
import subprocess, os

size_gb = os.path.getsize(model_path) / 1024**3
print(f"Model: {os.path.basename(model_path)}")
print(f"Size: {size_gb:.2f} GB")

r = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"]).decode()
print(f"VRAM (used/total): {r.strip()}")
print(f"Available for model: ~{15360 - int(r.split(',')[0].strip().replace('MiB','').strip())} MiB")

In [ ]:
# Start Cloudflare quick tunnel (ephemeral URL)
import subprocess, threading, time, re, sys

TUNNEL_URL = None

def run_tunnel(port=8080):
    global TUNNEL_URL
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(line, end="")
        m = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
        if m and not TUNNEL_URL:
            TUNNEL_URL = m.group(1)
            print(f"\n>>> TUNNEL URL: {TUNNEL_URL}")
            print(f">>> API endpoint: {TUNNEL_URL}/v1/chat/completions")
            print(f">>> Models: {TUNNEL_URL}/v1/models")

thread = threading.Thread(target=run_tunnel, daemon=True)
thread.start()

# Wait for tunnel URL
for _ in range(30):
    if TUNNEL_URL:
        break
    time.sleep(1)

if not TUNNEL_URL:
    print("WARNING: Tunnel URL not detected yet. Check cloudflared output above.")
else:
    print(f"\nReady! API at: {TUNNEL_URL}/v1/chat/completions")


In [ ]:
# Start the OpenAI-compatible API server in background
# Model loads ONLY here — not in the notebook kernel (saves 5-12GB VRAM)
import subprocess, os, signal

MODEL_PATH = model_path  # set by the download cell above
PORT = 8080

# Kill any existing server
os.system("pkill -f llama_server || true")

# Start llama-cpp-python server
server_script = f"""
import llama_cpp
from llama_cpp import Llama
import threading, json, http.server

llm = Llama(
    model_path="{MODEL_PATH}",
    n_gpu_layers=-1,
    n_ctx={CONTEXT},
    verbose=False,
    n_threads=2,
)

class APIHandler(http.server.BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/v1/models":
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(json.dumps({{"data":[{{"id":"{MODEL_NAME}"}}]}}).encode())
        else:
            self.send_response(404)
            self.end_headers()

    def do_POST(self):
        if self.path == "/v1/chat/completions":
            length = int(self.headers.get("Content-Length", 0))
            body = json.loads(self.rfile.read(length))
            messages = body.get("messages", [])
            prompt = ""
            for msg in messages:
                role = msg.get("role", "user")
                content = msg.get("content", "")
                prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"
            prompt += "<|im_start|>assistant\n"
            max_tokens = body.get("max_tokens", 256)
            temperature = body.get("temperature", 0.7)
            out = llm(prompt, max_tokens=max_tokens, temperature=temperature, stop=["<|im_end|>"])
            text = out["choices"][0]["text"]
            response = {{
                "id": "chatcmpl-1",
                "object": "chat.completion",
                "choices": [{{"index": 0, "message": {{"role": "assistant", "content": text}}, "finish_reason": "stop"}}],
                "usage": {{"prompt_tokens": 0, "completion_tokens": out["usage"]["completion_tokens"], "total_tokens": out["usage"]["completion_tokens"]}}
            }}
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(json.dumps(response).encode())
        else:
            self.send_response(404)
            self.end_headers()

    def log_message(self, format, *args):
        pass  # suppress logs

server = http.server.HTTPServer(("0.0.0.0", {PORT}), APIHandler)
print(f"Server running on port {PORT}")
server.serve_forever()
"""

with open("/tmp/server.py", "w") as f:
    f.write(server_script)

server_proc = subprocess.Popen(["python3", "/tmp/server.py"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Wait for server to start
import time
time.sleep(5)
print("Server started! Use the tunnel URL above.")
print(f"Test: curl {TUNNEL_URL}/v1/models")


In [ ]:
# Test the API
!curl -s {TUNNEL_URL}/v1/chat/completions \
  -X POST -H "Content-Type: application/json" \
  -d '{"messages":[{"role":"user","content":"What is 17 * 23? Think step by step."}],"max_tokens":200}'

print("\n\n--- Python client test ---")
from openai import OpenAI
client = OpenAI(base_url=f"{TUNNEL_URL}/v1", api_key="not-needed")
r = client.chat.completions.create(
    model="default",
    messages=[{"role": "user", "content": "Say hello in 3 languages."}],
    max_tokens=100
)
print(r.choices[0].message.content)
